# Rolling Windows Tutorial

## About Rolling Windows Analysis

Rolling windows analysis tracks how patterns change throughout a text by analyzing overlapping segments. It's like a moving spotlight that reveals trends, themes, and narrative arcs in documents. This technique is particularly powerful for analyzing literature, tracking sentiment over time, or studying how topics evolve in long texts.

A rolling window slides across your text, counting specified patterns in each segment. This creates a timeline showing how frequently your patterns appear throughout the document, revealing peaks, valleys, and trends that might be invisible when looking at the text as a whole.

## 1. Prerequisites

- Python ≥ 3.9
- `lexos` installed (see main README)
- spaCy English model: `python -m spacy download en_core_web_sm`

## 2. Creating Windows

You can create windows from different input types:

### 2.1 From spaCy Doc objects

In [ ]:
import spacy
from lexos.rolling_windows import Windows

nlp = spacy.load("en_core_web_sm")
doc = nlp("This is a sample text for analysis. It contains multiple sentences for demonstration.")

windows = Windows()

### 2.2 From raw text strings

In [ ]:
text = "Love was everywhere in the beginning. But sadness crept in later. Then love returned."
doc = nlp(text)

## 3. Generating Windows

In [ ]:
# Character-based windows
char_windows = windows(
    input=text,
    n=50,  # 50 characters per window
    window_type="characters",
    output="strings"
)

print("Character windows:")
for i, window in enumerate(list(char_windows)[:3]):
    print(f"Window {i+1}: '{window[:30]}...'")

In [ ]:
# Token-based windows (most common)
token_windows = windows(
    input=doc,
    n=10,  # 10 tokens per window
    window_type="tokens", 
    output="strings"
)

print("\nToken windows:")
for i, window in enumerate(list(token_windows)[:3]):
    print(f"Window {i+1}: {window}")

## 4. Analyzing Patterns with Calculators

Track specific patterns using calculators:

### 4.1 Counting occurrences

In [ ]:
from lexos.rolling_windows.calculators import Counts

# Create new windows for analysis
analysis_windows = windows(input=doc, n=8, window_type="tokens", output="strings")

# Count emotional words
emotion_counter = Counts(
    patterns=["love", "sadness", "happy", "joy"],
    windows=analysis_windows,
    mode="exact",
    case_sensitive=False
)

counts_df = emotion_counter.to_df()
print("Emotion word counts per window:")
print(counts_df.head())

### 4.2 Calculating averages

In [ ]:
from lexos.rolling_windows.calculators import Averages

# Create fresh windows (generators are consumed after use)
avg_windows = windows(input=doc, n=8, window_type="tokens", output="strings")

averages_calculator = Averages(
    patterns=["love", "sadness"],
    windows=avg_windows,
    mode="exact",
    case_sensitive=False
)

averages_df = averages_calculator.to_df()
print("Average frequency per window:")
print(averages_df)

### 4.3 Understanding calculator modes

In [ ]:
# Different matching modes
test_text = "I love lovely lovers and loving relationships"
test_doc = nlp(test_text)

# Exact matching - finds only "love"
exact_windows = windows(input=test_doc, n=15, window_type="tokens", output="strings")
exact_counts = Counts(patterns=["love"], windows=exact_windows, mode="exact")

# Regex matching - finds "love" within words
regex_windows = windows(input=test_doc, n=15, window_type="tokens", output="strings")
regex_counts = Counts(patterns=["love"], windows=regex_windows, mode="regex")

print(f"Exact matches: {exact_counts.to_df().iloc[0, 0]}")
print(f"Regex matches: {regex_counts.to_df().iloc[0, 0]}")

## 5. Visualizing Results

### 5.1 Simple matplotlib plots

In [ ]:
from lexos.rolling_windows.plotters import SimplePlotter
import matplotlib.pyplot as plt

plotter = SimplePlotter(
    title="Emotional Words Throughout Text",
    xlabel="Window Number",
    ylabel="Average Frequency",
    width=10,
    height=6
)

plotter(df=averages_df)
plt.show()

### 5.2 Interactive Plotly visualization

In [ ]:
from lexos.rolling_windows.plotters import PlotlyPlotter

interactive_plotter = PlotlyPlotter(
    title="Interactive Emotional Analysis",
    xlabel="Window Position",
    ylabel="Average Frequency",
    width=800,
    height=500
)

interactive_plotter(df=averages_df, show_plot=True)

### 5.3 Adding milestones for structure

In [ ]:
# Mark chapter/section boundaries
longer_text = """
Chapter 1: Love filled the air and happiness abounded everywhere.
Chapter 2: Sadness crept in as difficulties mounted and joy faded.
Chapter 3: Love returned stronger, bringing renewed happiness and hope.
"""

longer_doc = nlp(longer_text)

# Find chapter positions
chapter_positions = {}
for i, token in enumerate(longer_doc):
    if token.text == "Chapter":
        chapter_num = longer_doc[i+1].text if i+1 < len(longer_doc) else "?"
        chapter_positions[f"Ch {chapter_num}"] = i

print("Chapter positions:", chapter_positions)

In [ ]:
# Create analysis with milestones
milestone_windows = windows(input=longer_doc, n=15, window_type="tokens", output="strings")
milestone_averages = Averages(
    patterns=["love", "happiness", "sadness", "joy"],
    windows=milestone_windows,
    mode="exact",
    case_sensitive=False
)

# Plot with chapter markers
milestone_plotter = SimplePlotter(
    title="Emotional Journey with Chapter Markers",
    xlabel="Token Position",
    ylabel="Average Frequency",
    show_milestones=True,
    show_milestone_labels=True,
    milestone_labels=chapter_positions,
    width=12,
    height=7
)

milestone_plotter(df=milestone_averages.to_df())
plt.show()

## 6. Running the Test Suite

This module includes comprehensive tests with 100% coverage:

In [ ]:
averages = rw.result.plot.line()

In [ ]:
# Run all rolling windows tests
uv run pytest tests/rolling_windows/

# Run with coverage report
uv run pytest --cov=src/lexos/rolling_windows --cov-report=html tests/rolling_windows/


## Key Concepts Summary

- **Windows**: Overlapping text segments for analysis
- **Window Types**: `characters`, `tokens`, or `spans`
- **Calculators**: Tools to count patterns (`Counts`) or calculate averages (`Averages`)
- **Modes**: `exact` (precise matches), `regex` (pattern matching), `spacy_rule` (linguistic rules)
- **Plotters**: Visualization tools (`SimplePlotter` for static, `PlotlyPlotter` for interactive)
- **Milestones**: Structural markers (chapters, sections) for enhanced visualization